**Note: Try to avoid *GROUP BY* clause to solve the problems**

For the problems use the *Health Insurance Claim* dataset. You can get the details as well as the dataset from [here](https://www.kaggle.com/datasets/thedevastator/insurance-claim-analysis-demographic-and-health).

### **Problem 1:** What are the top 5 patients who claimed the highest insurance amounts?

### **Problem 2:** What is the average insurance claimed by patients based on the number of children they have?

### **Problem 3:** What is the highest and lowest claimed amount by patients in each region?

### **Problem 4:** What is the percentage of smokers in each age group?

### **Problem 5:** What is the difference between the claimed amount of each patient and the first claimed amount of that patient?

### **Problem 6:** For each patient, calculate the difference between their claimed amount and the average claimed amount of patients with the same number of children.

### **Problem 7:** Show the patient with the highest BMI in each region and their respective rank.

### **Problem 8:** Calculate the difference between the claimed amount of each patient and the claimed amount of the patient who has the highest BMI in their region.

### **Problem 9:** For each patient, calculate the difference in claim amount between the patient and the patient with the highest claim amount among patients with the same bmi and smoker status, within the same region. Return the result in descending order difference.

### **Problem 10:** For each patient, find the maximum BMI value among their next three records (ordered by age).

### **Problem 11:** For each patient, find the rolling average of the last 2 claims.

### **Problem 12:** Find the first claimed insurance value for male and female patients, within each region order the data by patient age in ascending order, and only include patients who are non-diabetic and have a bmi value between 25 and 30.

In [ ]:
-- Quick Tip: Use `DISTINCT ON (column_name) *` whenever there is a need to print
-- only the unique values belonging to a column in case of Window Functions.
-- An example is given in query 2.

-- 1.
SELECT index,
       patientid,
       claim,
       RANK() OVER(ORDER BY claim DESC)
FROM insurance_data;

-- 2.
SELECT DISTINCT ON (children) *
FROM (
    SELECT *,
       AVG(claim) OVER(PARTITION BY children) AS avg_claim
    FROM insurance_data
     ) AS sub_table;


-- 3.
SELECT *
FROM (
    SELECT *,
       DENSE_RANK() OVER(ORDER BY claim DESC) AS region_rank
    FROM insurance_data
     ) AS sub_table
WHERE region_rank = 1;

SELECT *
FROM (
    SELECT *,
       DENSE_RANK() OVER(ORDER BY claim) AS region_rank
    FROM insurance_data
     ) AS sub_table
WHERE region_rank = 1;

-- verify
SELECT *
FROM insurance_data
WHERE claim = (
    SELECT MIN(claim)
    FROM insurance_data
    );

-- 4.
SELECT age,
       (smoker_count * 100 / SUM(smoker_count) OVER()) AS smoker_percentage
FROM (
    SELECT age,
       COUNT(*) AS smoker_count
    FROM insurance_data
    WHERE smoker = 'Yes'
    GROUP BY age
     ) AS sub_table;
-- SUM(smoker_count) in line 2 of query 4 would normally raise an error due to
-- lack of group by.
-- So we simply avoid collapsing the rows in line 2 and include the sum of smoker_count
-- column with each row in the window using OVER().

-- 5.
WITH patients_claims AS (
    SELECT patientid,
       FIRST_VALUE(claim) over (PARTITION BY patientid) AS first_claim,
       SUM(claim) OVER(PARTITION BY patientid) AS total_claim
    FROM insurance_data
)
SELECT patientid,
       first_claim,
       total_claim,
       (total_claim - first_claim) AS claim_diff
FROM patients_claims;

-- 6.
WITH sub_table AS (
    SELECT patientid,
       children,
       claim,
       AVG(claim) OVER(PARTITION BY children) AS avg_claim
    FROM insurance_data
)
SELECT patientid,
       children,
       claim,
       avg_claim,
       (avg_claim - claim) AS claim_diff
FROM sub_table;

-- 7.
SELECT *,
       RANK() OVER(PARTITION BY region ORDER BY bmi DESC)
FROM insurance_data;

--8
WITH sub_table AS (
    SELECT patientid,
           region,
           claim,
           FIRST_VALUE(claim) over (PARTITION BY region ORDER BY bmi) AS highest_claim
    FROM insurance_data
    ORDER BY patientid
)
SELECT *,
       (highest_claim - claim) AS claim_diff
FROM sub_table;

-- 9.
WITH max_claims AS (
    SELECT
        patientid,
        claim,
        bmi,
        smoker,
        region,
        MAX(claim) OVER (PARTITION BY bmi, smoker, region) AS highest_claim_amount
    FROM insurance_data
)
SELECT
    patientid,
    claim,
    bmi,
    smoker,
    region,
    highest_claim_amount - claim AS claim_difference
FROM max_claims
ORDER BY claim_difference DESC;

-- 10.
SELECT *,
       MAX(bmi) OVER(
           ORDER BY age ASC NULLS LAST
           ROWS BETWEEN 1 FOLLOWING AND 3 FOLLOWING
       ) AS max_bmi_next_three
FROM insurance_data;

-- 11.
SELECT
    patientid,
    index,
    claim,
    AVG(claim) OVER (
        PARTITION BY patientid
        ORDER BY index
        ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
    ) AS rolling_avg_last_2
FROM insurance_data
ORDER BY patientid, index;

-- 12.
WITH filtered_patients AS (
    SELECT
        region,
        gender,
        age,
        bmi,
        diabetic,
        claim,
        ROW_NUMBER() OVER (
            PARTITION BY region, gender
            ORDER BY age ASC, index ASC
        ) AS rn
    FROM insurance_data
    WHERE diabetic = 'No'
      AND bmi BETWEEN 25 AND 30
)
SELECT
    region,
    gender,
    age,
    bmi,
    claim AS first_claim_amount
FROM filtered_patients
WHERE rn = 1
ORDER BY region, gender, age ASC;